<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2024/2025  

<img src="../../img/portada.png" width="300"/>

---

#  Ensamble Secuencial de Modelos Predictivos  
### Primera Convocatoria – Junio 2025  

**Profesor:** Juan Galán Páez  
**Autores:**  
Carlos Martín de Prado Barragán  
Marco Padilla Gómez  

**Fecha de entrega:** 9 de Junio de 2025  

</div>

## Introducción

En este notebook se realiza una exploración exhaustiva de los hiperparámetros del meta-modelo secuencial aplicado al conjunto de datos sobre los precios de la vivienda.

Mediante validación cruzada con 5 particiones, se evalúan diferentes combinaciones de parámetros clave como el número de estimadores, la tasa de aprendizaje y la profundidad máxima de los árboles base.

Además del coeficiente de determinación R², se incluyen métricas complementarias como el Error Absoluto Medio (MAE) y la Raíz del Error Cuadrático Medio (RMSE) para proporcionar una evaluación más completa del rendimiento.

El objetivo es identificar las configuraciones que maximicen la capacidad predictiva y la robustez del modelo, fundamentando así las decisiones de diseño y optimización.

## 1. Importación de librerías y configuración del entorno

- Se añade la ruta `../../src` para importar el módulo personalizado `sequential_ensemble.py` que contiene la implementación del meta-modelo secuencial.
- Se importan las librerías necesarias para el procesamiento de datos (`numpy`, `pandas`), modelado (`DecisionTreeRegressor`), preprocesamiento (`StandardScaler`, `OneHotEncoder`), validación cruzada (`cross_validate`, `KFold`), y métricas de evaluación (`mean_absolute_error`, `mean_squared_error`).
- Finalmente, se importa la clase `SequentialEnsembleRegressor` para utilizar el meta-modelo en los experimentos.

In [5]:
# Importación de librerías necesarias
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.compose import ColumnTransformer

# Importar el meta-modelo desde nuestro módulo
import sys
sys.path.append('../../src')

from sequential_ensemble import SequentialEnsembleRegressor

## 2. Carga y preprocesamiento del dataset

- Se carga el conjunto de datos `house_prices.csv` desde la carpeta `data`.
- Se separan las variables predictoras (`X`) y la variable objetivo (`y`), que corresponde a la escala `SalePrice`.
- Se aplica estandarización a las variables predictoras utilizando `StandardScaler` para normalizar su escala y facilitar el entrenamiento del modelo y `OneHotEncoder` para codificar las variables categóricas.
- Se combinan ambas transformaciones con un `ColumnTransformer`.

In [6]:
# Cargar el dataset de Parkinson
df = pd.read_csv("../../data/house_prices.csv")

# Separar variables predictoras y variable objetivo
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

# Detectar variables categóricas (tipo objeto) y numéricas
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

# Pipeline de preprocesamiento
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

# Aplicar el preprocesamiento al conjunto de datos
X_processed = preprocessor.fit_transform(X)

## 3. Definición del espacio de búsqueda de hiperparámetros

- Se establecen los rangos de valores para los hiperparámetros clave a explorar:
  - `n_estimators`: número de modelos base en el ensamblador, con valores 10, 30 y 50.
  - `learning_rate`: tasa de aprendizaje que modula la contribución de cada modelo, con valores 0.05, 0.1 y 0.2.
  - `max_depth`: profundidad máxima permitida para los árboles de decisión, con valores 3, 4 y 5.
  - `sample_size`: porcentaje del conjunto de entrenamiento en cada iteración.
- Se inicializa una lista vacía `results` para almacenar los resultados de cada combinación probada durante la experimentación.

In [15]:
# Definimos rangos para los hiperparámetros más relevantes
n_estimators_options = [10, 30, 50]
learning_rate_options = [0.05, 0.1, 0.2]
max_depth_options = [3, 4, 5]
sample_size_options = [0.6, 0.75, 0.9]  

# Para guardar resultados
results = []

## 4. Ejecución de experimentos con validación cruzada

- Se configura la validación cruzada con 5 particiones (`KFold`), mezclando aleatoriamente los datos para mayor representatividad.
- Se realiza un bucle anidado para probar todas las combinaciones posibles de los hiperparámetros definidos: número de estimadores, tasa de aprendizaje, profundidad máxima y porcentaje del conjunto de entrenamiento.
- Para cada configuración, se crea una instancia del meta-modelo secuencial con esos parámetros.
- Se ejecuta la validación cruzada calculando múltiples métricas:  
  - Coeficiente de determinación R²,  
  - Error Absoluto Medio (MAE),  
  - Raíz del Error Cuadrático Medio (RMSE).
- Los resultados (medias y desviaciones estándar) se almacenan en la lista `results` para su posterior análisis.
- Se imprime un resumen en cada iteración para seguimiento en tiempo real de los experimentos.

In [16]:
# Configurar validación cruzada
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Métricas a evaluar como diccionario para evitar NaN
scoring = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}

# Bucle para probar combinaciones con validación cruzada
for sample_size in sample_size_options:
    for n_estimators in n_estimators_options:
        for lr in learning_rate_options:
            for max_depth in max_depth_options:
                modelo = SequentialEnsembleRegressor(
                    base_estimator=DecisionTreeRegressor,
                    n_estimators=n_estimators,
                    lr=lr,
                    sample_size=sample_size,
                    est_params={"max_depth": max_depth},
                    random_state=42
                )
                scores = cross_validate(modelo, X_processed, y, cv=kf, scoring=scoring)

                mean_r2 = np.mean(scores['test_r2'])
                mean_mae = -np.mean(scores['test_mae'])
                mean_rmse = np.sqrt(-np.mean(scores['test_mse']))
                std_r2 = np.std(scores['test_r2'])

                results.append({
                    "sample_size": sample_size,
                    "n_estimators": n_estimators,
                    "learning_rate": lr,
                    "max_depth": max_depth,
                    "mean_r2": mean_r2,
                    "std_r2": std_r2,
                    "mean_mae": mean_mae,
                    "mean_rmse": mean_rmse
                })

                print(f"Evaluado sample_size={sample_size}, n_estimators={n_estimators}, "
                      f"lr={lr}, max_depth={max_depth} -> R2 mean: {mean_r2:.4f}")

Evaluado sample_size=0.6, n_estimators=10, lr=0.05, max_depth=3 -> R2 mean: -1.5483
Evaluado sample_size=0.6, n_estimators=10, lr=0.05, max_depth=4 -> R2 mean: -1.5276
Evaluado sample_size=0.6, n_estimators=10, lr=0.05, max_depth=5 -> R2 mean: -1.5227
Evaluado sample_size=0.6, n_estimators=10, lr=0.1, max_depth=3 -> R2 mean: -0.0715
Evaluado sample_size=0.6, n_estimators=10, lr=0.1, max_depth=4 -> R2 mean: -0.0470
Evaluado sample_size=0.6, n_estimators=10, lr=0.1, max_depth=5 -> R2 mean: -0.0210
Evaluado sample_size=0.6, n_estimators=10, lr=0.2, max_depth=3 -> R2 mean: 0.6565
Evaluado sample_size=0.6, n_estimators=10, lr=0.2, max_depth=4 -> R2 mean: 0.6689
Evaluado sample_size=0.6, n_estimators=10, lr=0.2, max_depth=5 -> R2 mean: 0.6707
Evaluado sample_size=0.6, n_estimators=30, lr=0.05, max_depth=3 -> R2 mean: 0.4343
Evaluado sample_size=0.6, n_estimators=30, lr=0.05, max_depth=4 -> R2 mean: 0.4476
Evaluado sample_size=0.6, n_estimators=30, lr=0.05, max_depth=5 -> R2 mean: 0.4510
Eval

## 5. Análisis y visualización de resultados

- Se convierte la lista de resultados en un DataFrame de Pandas para facilitar su manipulación.
- Se ordenan las configuraciones probadas según el valor medio del coeficiente de determinación R², de mayor a menor, para identificar rápidamente las mejores combinaciones.
- Se reinicia el índice del DataFrame para mantener la consistencia tras el ordenamiento.
- Finalmente, se muestra la tabla con todas las configuraciones evaluadas y sus respectivos resultados, facilitando el análisis comparativo.

In [17]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="mean_r2", ascending=False)
results_df.reset_index(drop=True, inplace=True)
results_df

,sample_size,n_estimators,learning_rate,max_depth,mean_r2,std_r2,mean_mae,mean_rmse
0,0.75,50,0.10,3,0.767748,0.039509,23520.752716,37604.916182
1,0.60,50,0.10,5,0.766624,0.044999,22654.988830,37886.824878
2,0.75,50,0.10,4,0.762110,0.044091,23853.979976,38158.632992
3,0.60,50,0.10,4,0.760338,0.028210,23827.677460,38143.257741
4,0.60,50,0.20,3,0.757761,0.054642,23819.232315,37904.915075
...,...,...,...,...,...,...,...,...
76,0.90,10,0.05,5,-1.528553,0.360996,106443.458536,121644.164822
77,0.75,10,0.05,3,-1.539149,0.348489,106275.972415,121992.470958
78,0.90,10,0.05,4,-1.542475,0.334322,106611.586496,122142.703451
79,0.60,10,0.05,3,-1.548262,0.350886,106600.499367,122153.919563


In [18]:
# Comparación del mejor ensamblador con el modelo base individual (DecisionTreeRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

# Extraer la mejor configuración del ensamblador
mejor_config = results_df.iloc[0]
mejor_max_depth = int(mejor_config["max_depth"])

# Crear y evaluar el modelo base individual con ese max_depth
modelo_base = DecisionTreeRegressor(max_depth=mejor_max_depth, random_state=42)
r2_base = np.mean(cross_val_score(modelo_base, X_processed, y, cv=kf, scoring='r2'))

# Mostrar comparación
print(f"R² del mejor ensamblador (validación cruzada): {mejor_config['mean_r2']:.4f}")
print(f"R² del modelo base individual (max_depth={mejor_max_depth}): {r2_base:.4f}")

R² del mejor ensamblador (validación cruzada): 0.7677
R² del modelo base individual (max_depth=3): 0.4488


## 6. Interpretación de resultados y conclusiones

Los experimentos realizados evidencian que el rendimiento del meta-modelo secuencial está fuertemente condicionado por la configuración de los hiperparámetros utilizados.

- Cuando se emplea un número reducido de estimadores (`n_estimators=10`) y una tasa de aprendizaje baja (`lr=0.05`), el modelo obtiene valores negativos de R², lo que indica que su desempeño es **peor que una predicción constante basada en la media**. Esto sugiere una capacidad insuficiente del modelo para capturar patrones significativos en los datos bajo estas condiciones.

- A medida que se incrementa el número de estimadores a 50 y se ajustan los valores de `learning_rate`, se observa una mejora significativa en el rendimiento. La mejor configuración encontrada fue con:
    - `sample_size=0.75`
    - `n_estimators=50`
    - `learning_rate=0.10`
    - `max_depth=3`

    alcanzando un R² promedio de 0.7677, lo que indica que el modelo es capaz de explicar aproximadamente el 76.8% de la variabilidad en los precios de vivienda.

- La **profundidad máxima (`max_depth`)** también juega un papel importante. Profundidades moderadas como 3 o 4 han dado buenos resultados, equilibrando la capacidad de aprendizaje sin incurrir en sobreajuste.

- La **tasa de aprendizaje (`lr`)**, los valores intermedios como 0.1 parecen ofrecer un buen compromiso entre velocidad de convergencia y estabilidad. Valores más altos o bajos sin el acompañamiento adecuado de otros hiperparámetros tienden a generar resultados subóptimos.

- El parámetro **`sample_size`**, ha tenido un comporttamiento curioso debido a que los mejores resultados se han obtenido con los valores medios y bajos.

- Las métricas de error absolutas (`mean_mae` y `mean_rmse`) complementan esta interpretación. Las configuraciones con mejor R² presentan también los menores errores, con **MAE en torno a 23,500 y RMSE alrededor de 38,000**, lo que representa errores relativos aceptables considerando que el precio medio de las casas está en el orden de los 180,000 a 200,000.

- Finalmente, se realizó una comparación directa entre el mejor modelo ensamblador y un modelo individual `DecisionTreeRegressor` con `max_depth=3`. El ensamblador alcanzó un R² promedio de **0.7677**, mientras que el modelo base obtuvo solo **0.4488**. Esta diferencia confirma que el **aprendizaje secuencial con corrección de errores** permite construir un modelo con **mayor capacidad predictiva** que un único estimador tradicional.